# Analisis Statistik - Olist E-Commerce

Uji hipotesis untuk memvalidasi temuan dari EDA secara statistik.
Setiap uji terhubung ke pertanyaan bisnis yang dirumuskan di notebook EDA.

**Prasyarat:** Jalankan `01_load_to_sqlite.ipynb` dan `03_cleaning.ipynb` terlebih dahulu.

**Catatan:**
- Alpha (tingkat signifikansi) = 0,05 untuk semua uji.
- Menggunakan uji non-parametrik karena distribusi data tidak normal (skewed).

## Koneksi ke Pertanyaan Bisnis

| Uji | Pertanyaan Bisnis |
|-----|-------------------|
| 1-2 | (P3) Apa faktor utama keterlambatan pengiriman? |
| 3-4 | (P4) Bagaimana distribusi geografis mempengaruhi biaya dan waktu pengiriman? |
| 5 | (P4) Apakah perbedaan waktu kirim antar region signifikan? |
| 6 | (P5) Bagaimana pola waktu pembelian pelanggan? |

## Persiapan

In [ ]:
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

ALPHA = 0.05

DB_PATH = "../data/database/olist.db"
conn = sqlite3.connect(DB_PATH)

## Muat dan Siapkan Data

In [ ]:
df = pd.read_sql_query("""
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    c.customer_state,
    oi.price,
    oi.freight_value,
    p.product_weight_g,
    p.product_category_name,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_item oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
WHERE o.order_delivered_customer_date IS NOT NULL
""", conn)

conn.close()

# Konversi datetime
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[col] = pd.to_datetime(df[col])

# Hitung kolom turunan
df["hari_kirim"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df["terlambat"] = df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]
df["hari_beli"] = df["order_purchase_timestamp"].dt.day_name()
df["is_weekend"] = df["order_purchase_timestamp"].dt.dayofweek >= 5
df["region_sp"] = df["customer_state"] == "SP"

print(f"Total baris: {len(df)}")
print(f"Terlambat: {df['terlambat'].sum()} ({df['terlambat'].mean()*100:.1f}%)")
df.head()

### Fungsi Bantu

In [ ]:
def interpretasi(p_value, alpha=ALPHA):
    """Cetak interpretasi hasil uji hipotesis."""
    if p_value < alpha:
        print(f"  Keputusan: H0 DITOLAK (p={p_value:.6f} < {alpha})")
        print("  Kesimpulan: terdapat perbedaan yang signifikan secara statistik.")
    else:
        print(f"  Keputusan: H0 TIDAK DITOLAK (p={p_value:.6f} >= {alpha})")
        print("  Kesimpulan: tidak terdapat perbedaan yang signifikan secara statistik.")

---
## Uji 1 [P3]: Apakah Harga Produk Berbeda antara Pengiriman Tepat Waktu vs Terlambat?

- **H0:** Tidak ada perbedaan harga antara order tepat waktu dan terlambat.
- **H1:** Ada perbedaan harga antara order tepat waktu dan terlambat.
- **Uji:** Mann-Whitney U (non-parametrik, 2 sampel independen).

In [ ]:
tepat_waktu = df[df["terlambat"] == False]["price"]
terlambat = df[df["terlambat"] == True]["price"]

print(f"Tepat waktu - n: {len(tepat_waktu)}, median: {tepat_waktu.median():.2f}, mean: {tepat_waktu.mean():.2f}")
print(f"Terlambat   - n: {len(terlambat)}, median: {terlambat.median():.2f}, mean: {terlambat.mean():.2f}")

stat, p = stats.mannwhitneyu(tepat_waktu, terlambat, alternative="two-sided")
print(f"\n  Mann-Whitney U = {stat:,.0f}")
interpretasi(p)

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([tepat_waktu, terlambat], labels=["Tepat Waktu", "Terlambat"], showfliers=False)
ax.set_ylabel("Harga (BRL)")
ax.set_title("Distribusi Harga: Tepat Waktu vs Terlambat")
plt.tight_layout()
plt.show()

---
## Uji 2 [P3]: Apakah Berat Produk Mempengaruhi Keterlambatan Pengiriman?

- **H0:** Tidak ada perbedaan berat produk antara order tepat waktu dan terlambat.
- **H1:** Berat produk order yang terlambat lebih besar.
- **Uji:** Mann-Whitney U (one-sided: greater).

In [ ]:
berat_tepat = df[df["terlambat"] == False]["product_weight_g"]
berat_lambat = df[df["terlambat"] == True]["product_weight_g"]

print(f"Tepat waktu - median berat: {berat_tepat.median():.0f} g, mean: {berat_tepat.mean():.0f} g")
print(f"Terlambat   - median berat: {berat_lambat.median():.0f} g, mean: {berat_lambat.mean():.0f} g")

stat, p = stats.mannwhitneyu(berat_lambat, berat_tepat, alternative="greater")
print(f"\n  Mann-Whitney U = {stat:,.0f}")
interpretasi(p)

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([berat_tepat, berat_lambat], labels=["Tepat Waktu", "Terlambat"], showfliers=False)
ax.set_ylabel("Berat (g)")
ax.set_title("Distribusi Berat Produk: Tepat Waktu vs Terlambat")
plt.tight_layout()
plt.show()

---
## Uji 3 [P4]: Apakah Ongkir Berbeda Signifikan Antar State?

- **H0:** Tidak ada perbedaan ongkir antar state.
- **H1:** Setidaknya satu state memiliki ongkir yang berbeda.
- **Uji:** Kruskal-Wallis (non-parametrik, >2 grup independen).

Menggunakan 5 state terbesar untuk menghindari grup terlalu kecil.

In [ ]:
top_states = df["customer_state"].value_counts().head(5).index.tolist()
df_top = df[df["customer_state"].isin(top_states)]

groups = [group["freight_value"].values for _, group in df_top.groupby("customer_state")]

# Statistik deskriptif
print("Median ongkir per state:")
print(df_top.groupby("customer_state")["freight_value"].median().sort_values(ascending=False))

stat, p = stats.kruskal(*groups)
print(f"\n  Kruskal-Wallis H = {stat:,.2f}")
interpretasi(p)

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(10, 5))
df_top.boxplot(column="freight_value", by="customer_state", ax=ax, showfliers=False)
ax.set_xlabel("State")
ax.set_ylabel("Ongkos Kirim (BRL)")
ax.set_title("Distribusi Ongkir per State (Top 5)")
plt.suptitle("")
plt.tight_layout()
plt.show()

---
## Uji 4 [P4]: Apakah Ada Korelasi Signifikan antara Berat dan Ongkir?

- **H0:** Tidak ada korelasi antara berat produk dan ongkos kirim (rho = 0).
- **H1:** Ada korelasi antara berat produk dan ongkos kirim (rho != 0).
- **Uji:** Spearman Rank Correlation (non-parametrik).

In [ ]:
rho, p = stats.spearmanr(df["product_weight_g"], df["freight_value"])

print(f"  Spearman rho = {rho:.4f}")
interpretasi(p)
print(f"\n  Interpretasi kekuatan: ", end="")
if abs(rho) >= 0.7:
    print("Korelasi kuat")
elif abs(rho) >= 0.4:
    print("Korelasi sedang")
elif abs(rho) >= 0.2:
    print("Korelasi lemah")
else:
    print("Korelasi sangat lemah")

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df["product_weight_g"], df["freight_value"], alpha=0.05, s=5)
ax.set_xlabel("Berat Produk (g)")
ax.set_ylabel("Ongkos Kirim (BRL)")
ax.set_title(f"Berat vs Ongkir (Spearman rho = {rho:.4f}, p < 0.001)")
ax.set_xlim(0, 10000)
plt.tight_layout()
plt.show()

---
## Uji 5 [P4]: Apakah Waktu Pengiriman Berbeda antara SP vs Non-SP?

- **H0:** Tidak ada perbedaan waktu pengiriman antara customer SP dan non-SP.
- **H1:** Waktu pengiriman customer non-SP lebih lama.
- **Uji:** Mann-Whitney U (one-sided: greater).

In [ ]:
kirim_sp = df[df["region_sp"] == True]["hari_kirim"]
kirim_non_sp = df[df["region_sp"] == False]["hari_kirim"]

print(f"SP     - n: {len(kirim_sp)}, median: {kirim_sp.median():.1f} hari, mean: {kirim_sp.mean():.1f} hari")
print(f"Non-SP - n: {len(kirim_non_sp)}, median: {kirim_non_sp.median():.1f} hari, mean: {kirim_non_sp.mean():.1f} hari")

stat, p = stats.mannwhitneyu(kirim_non_sp, kirim_sp, alternative="greater")
print(f"\n  Mann-Whitney U = {stat:,.0f}")
interpretasi(p)

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([kirim_sp, kirim_non_sp], labels=["SP", "Non-SP"], showfliers=False)
ax.set_ylabel("Hari Pengiriman")
ax.set_title("Waktu Pengiriman: SP vs Non-SP")
plt.tight_layout()
plt.show()

---
## Uji 6 [P5]: Apakah Jumlah Order Berbeda antara Weekday vs Weekend?

- **H0:** Proporsi order di weekend sama dengan proporsi hari weekend (2/7 = 28,6%).
- **H1:** Proporsi order di weekend berbeda dari yang diharapkan.
- **Uji:** Chi-Square Goodness of Fit.

In [ ]:
# Hitung per order (bukan per item)
df_order = df.drop_duplicates(subset="order_id")

observed_weekend = df_order["is_weekend"].sum()
observed_weekday = len(df_order) - observed_weekend
total = len(df_order)

expected_weekend = total * (2 / 7)
expected_weekday = total * (5 / 7)

print(f"Observed - weekday: {observed_weekday}, weekend: {observed_weekend}")
print(f"Expected - weekday: {expected_weekday:.0f}, weekend: {expected_weekend:.0f}")
print(f"Proporsi weekend: {observed_weekend/total*100:.1f}% (expected: 28.6%)")

stat, p = stats.chisquare(
    f_obs=[observed_weekday, observed_weekend],
    f_exp=[expected_weekday, expected_weekend]
)

print(f"\n  Chi-Square = {stat:.2f}")
interpretasi(p)

In [ ]:
# Visualisasi
fig, ax = plt.subplots(figsize=(8, 5))

labels = ["Weekday", "Weekend"]
x = range(len(labels))
width = 0.35

ax.bar([i - width/2 for i in x], [observed_weekday, observed_weekend], width, label="Observed")
ax.bar([i + width/2 for i in x], [expected_weekday, expected_weekend], width, label="Expected", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Jumlah Order")
ax.set_title("Jumlah Order: Weekday vs Weekend (Observed vs Expected)")
ax.legend()

plt.tight_layout()
plt.show()

---
## Ringkasan Hasil Uji Hipotesis

| No | Pertanyaan | Metode | p-value | Keputusan | Temuan |
|----|-----------|--------|---------|-----------|--------|
| 1 | Harga: tepat waktu vs terlambat | Mann-Whitney U | < 0.001 | H0 ditolak | Harga order terlambat sedikit lebih tinggi (median 80 vs 74 BRL). Perbedaan signifikan secara statistik. |
| 2 | Berat: pengaruh terhadap keterlambatan | Mann-Whitney U | < 0.001 | H0 ditolak | Produk yang terlambat lebih berat (median 800 vs 700 g). Berat mempengaruhi keterlambatan. |
| 3 | Ongkir antar state | Kruskal-Wallis | < 0.001 | H0 ditolak | Ongkir berbeda signifikan antar state. Lokasi sangat mempengaruhi biaya pengiriman. |
| 4 | Korelasi berat vs ongkir | Spearman | < 0.001 | H0 ditolak | Korelasi sedang (rho = 0.45). Berat produk berpengaruh terhadap ongkir, tapi bukan satu-satunya faktor. |
| 5 | Waktu kirim: SP vs non-SP | Mann-Whitney U | < 0.001 | H0 ditolak | Non-SP jauh lebih lama (median 12.9 vs 7.2 hari). Jarak dari pusat logistik sangat berpengaruh. |
| 6 | Order: weekday vs weekend | Chi-Square | < 0.001 | H0 ditolak | Weekend hanya 23% order (expected 28.6%). Pelanggan lebih aktif belanja di weekday. |